# Sesión 06 — Clasificadores Lineales
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo II · Modelos Discriminativos**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Derivar la función de pérdida de entropía cruzada binaria desde el modelo probabilístico de regresión logística.
2. Implementar descenso de gradiente por lotes (GD) y mini-lotes (SGD) desde cero y comparar sus curvas de convergencia.
3. Implementar el optimizador Adam desde cero y explicar el papel de los momentos de primer y segundo orden.
4. Visualizar las trayectorias de regularización L1 y L2 y explicar por qué L1 induce dispersión.
5. Extender la regresión logística al caso multiclase mediante la función softmax.
6. Diagnosticar subajuste y sobreajuste mediante curvas de aprendizaje.

## Conjunto de datos principal

**PhysioNet Challenge 2012 — Mortalidad en UCI**  
Silva, I. et al. (2012). Predicting in-hospital mortality of ICU patients: the PhysioNet/Computing in Cardiology Challenge 2012.  
*Computing in Cardiology*, 39, 245–248.  
https://physionet.org/content/challenge-2012/  
Usamos un subconjunto simulado con la misma distribución de características reportada en el challenge.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Bishop (2006). *PRML*. §4.3 (Regresión logística), §3.3 (Regularización bayesiana). Springer. |
| ★★★ | Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2ª ed.). §4.4. Springer. |
| ★★☆ | Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *JRSS-B*, 58(1), 267–288. |
| ★★☆ | Kingma, D.P. & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR 2015*. https://arxiv.org/abs/1412.6980 |
| ★☆☆ | Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. Cap. 8 (Optimización). MIT Press. deeplearningbook.org |

## Parte 0 — Configuración y datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# ── Dataset: PhysioNet CinC 2012 — Mortalidad en UCI (simulado) ───────────────
# 8 características clínicas extraídas durante las primeras 48 h en UCI:
# edad, APACHE II, FC media, SpO2 media, creatinina, bilirrubina,
# Glasgow Coma Scale, fracción de inspiración de O2
# Fuente de referencia:
# Silva, I. et al. (2012). Predicting in-hospital mortality of ICU patients.
# Computing in Cardiology, 39, 245–248.
# https://physionet.org/content/challenge-2012/

N       = 800
N_feats = 8
nombres_feats = [
    'Edad', 'APACHE-II', 'FC media', 'SpO2 media',
    'Creatinina', 'Bilirrubina', 'Glasgow', 'FiO2'
]

def generar_uci(N, rng_):
    """Genera un dataset UCI simulado con distribución realista."""
    edad      = rng_.normal(63, 16, N).clip(18, 95)
    apache    = rng_.normal(18, 8,  N).clip(0, 71)
    fc        = rng_.normal(88, 20, N).clip(40, 180)
    spo2      = rng_.normal(94, 5,  N).clip(60, 100)
    creat     = rng_.gamma(2, 0.8,  N).clip(0.3, 15)
    bili      = rng_.gamma(1.5, 0.6, N).clip(0.1, 20)
    glasgow   = rng_.normal(11, 4,  N).clip(3, 15)
    fio2      = rng_.beta(2, 5, N) * 0.6 + 0.21

    X = np.column_stack([edad, apache, fc, spo2, creat, bili, glasgow, fio2])

    # Probabilidad de mortalidad (logit lineal)
    logit = (-4.5
             + 0.03 * edad
             + 0.12 * apache
             + 0.008 * fc
             - 0.05 * spo2
             + 0.15 * creat
             + 0.08 * bili
             - 0.10 * glasgow
             + 2.0  * fio2)
    p_muerte = 1 / (1 + np.exp(-logit))
    y = rng_.binomial(1, p_muerte).astype(float)
    return X.astype(np.float32), y

X_uci, y_uci = generar_uci(N, rng)

# Normalización z-score
mu_uci  = X_uci.mean(axis=0)
std_uci = X_uci.std(axis=0) + 1e-8
X_uci_s = (X_uci - mu_uci) / std_uci

# División train/test estratificada (75/25)
idx_pos = np.where(y_uci == 1)[0]
idx_neg = np.where(y_uci == 0)[0]
n_tr_p  = int(len(idx_pos) * 0.75)
n_tr_n  = int(len(idx_neg) * 0.75)
idx_tr  = np.concatenate([rng.permutation(idx_pos)[:n_tr_p],
                            rng.permutation(idx_neg)[:n_tr_n]])
idx_te  = np.setdiff1d(np.arange(N), idx_tr)

X_tr, y_tr = X_uci_s[idx_tr], y_uci[idx_tr]
X_te, y_te = X_uci_s[idx_te], y_uci[idx_te]

prev = y_uci.mean()
print(f'Dataset UCI: N={N}, características={N_feats}')
print(f'Prevalencia mortalidad: {prev:.1%}')
print(f'Train: {len(X_tr)} | Test: {len(X_te)}')

## Parte 1 — Regresión logística: derivación de la pérdida

El modelo de regresión logística asume:

$$P(y=1 \mid \mathbf{x}, \mathbf{w}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1+e^{-(\mathbf{w}^\top \mathbf{x}+b)}}$$

La función de pérdida de **entropía cruzada binaria** se obtiene tomando el negativo del log de la verosimilitud:

$$\mathcal{L}(\mathbf{w}) = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i) \right]$$

El gradiente (necesario para el descenso de gradiente) es:

$$\nabla_\mathbf{w} \mathcal{L} = \frac{1}{N}\mathbf{X}^\top(\hat{\mathbf{p}} - \mathbf{y})$$

Nótese la elegancia: el gradiente es simplemente el error de predicción ponderado por las características.

In [ ]:
# ── Regresión logística implementada desde cero ───────────────────────────────
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def entropia_cruzada(y, p_hat, eps=1e-9):
    p_hat = np.clip(p_hat, eps, 1 - eps)
    return -np.mean(y * np.log(p_hat) + (1 - y) * np.log(1 - p_hat))

def gradiente_logistico(X, y, w, b):
    N     = len(y)
    p_hat = sigmoid(X @ w + b)
    error = p_hat - y
    grad_w = X.T @ error / N
    grad_b = error.mean()
    return grad_w, grad_b, p_hat

def auroc_rapido(y_true, scores):
    """AUROC sin scipy, para uso en el loop de entrenamiento."""
    idx_pos = y_true == 1
    idx_neg = y_true == 0
    if idx_pos.sum() == 0 or idx_neg.sum() == 0:
        return 0.5
    return np.mean([np.mean(scores[idx_pos] > s) for s in scores[idx_neg]])

# Visualizar la superficie de pérdida en 2D (dos características)
w1_grid = np.linspace(-3, 3, 80)
w2_grid = np.linspace(-3, 3, 80)
W1, W2  = np.meshgrid(w1_grid, w2_grid)

# Usar solo las 2 características más informativas para visualización
X2d = X_tr[:, [1, 0]]   # APACHE-II y Edad

L_surf = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        w_ij  = np.array([W1[i,j], W2[i,j]])
        p_hat = sigmoid(X2d @ w_ij)
        L_surf[i,j] = entropia_cruzada(y_tr, p_hat)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cf = axes[0].contourf(W1, W2, L_surf, levels=25, cmap='viridis')
plt.colorbar(cf, ax=axes[0], label='Pérdida (entropía cruzada)')
axes[0].set(xlabel='w₁ (APACHE-II)', ylabel='w₂ (Edad)',
            title='Superficie de pérdida — regresión logística\n'
                  '(2 características, sesgo=0 fijado para visualización)')

# Curva sigmoide y su relación con la probabilidad
z = np.linspace(-6, 6, 300)
axes[1].plot(z, sigmoid(z), 'steelblue', lw=2.5)
axes[1].axhline(0.5, color='gray', ls='--', lw=1)
axes[1].axvline(0,   color='gray', ls='--', lw=1)
axes[1].fill_between(z, 0, sigmoid(z), where=(z > 0),
                      alpha=0.15, color='tomato', label='Predicho: muere (ŷ=1)')
axes[1].fill_between(z, 0, sigmoid(z), where=(z < 0),
                      alpha=0.15, color='steelblue', label='Predicho: sobrevive (ŷ=0)')
axes[1].set(xlabel='$z = \\mathbf{w}^\\top\\mathbf{x} + b$',
            ylabel='$\\sigma(z) = P(y=1|\\mathbf{x})$',
            title='Función sigmoide — mapea logit a probabilidad')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Parte 2 — Descenso de gradiente: GD, SGD y Adam

Comparamos tres optimizadores implementados desde cero:

| Optimizador | Actualización | Ventaja |
|---|---|---|
| **GD (lotes completos)** | $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla \mathcal{L}$ | Convergencia suave |
| **SGD (mini-lotes)** | $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla \mathcal{L}_{\text{batch}}$ | Más rápido por época |
| **Adam** | Momento adaptativo de primer y segundo orden | Robusto a elección de $\eta$ |

In [ ]:
# ── GD, SGD y Adam desde cero ─────────────────────────────────────────────────
def entrenar_logistico(X_tr, y_tr, X_te, y_te,
                        optimizador='gd', lr=0.1, n_epochs=150,
                        batch_size=64, lam=0.0, penalizacion='l2',
                        rng_=None):
    """
    Entrena regresión logística con GD, SGD o Adam.
    Retorna historiales de pérdida y AUROC.
    """
    rng_ = rng_ or np.random.default_rng()
    N, d = X_tr.shape
    w = np.zeros(d)
    b = 0.0

    # Variables de momento para Adam
    m_w, v_w = np.zeros(d), np.zeros(d)
    m_b, v_b = 0.0, 0.0
    beta1, beta2, eps_adam = 0.9, 0.999, 1e-8
    t_adam = 0

    hist = {'perdida_tr': [], 'perdida_te': [], 'auroc_te': []}

    for epoch in range(n_epochs):
        if optimizador == 'gd':
            batches = [(np.arange(N), y_tr)]
        else:
            idx_sh = rng_.permutation(N)
            batches = [(idx_sh[i:i+batch_size], y_tr[idx_sh[i:i+batch_size]])
                        for i in range(0, N, batch_size)]

        for idx_b, y_b in batches:
            X_b = X_tr[idx_b]
            gw, gb, _ = gradiente_logistico(X_b, y_b, w, b)

            # Regularización
            if lam > 0:
                if penalizacion == 'l2':
                    gw += lam * w
                else:  # l1
                    gw += lam * np.sign(w)

            if optimizador in ('gd', 'sgd'):
                w -= lr * gw
                b -= lr * gb
            else:  # adam
                t_adam += 1
                m_w = beta1*m_w + (1-beta1)*gw
                v_w = beta2*v_w + (1-beta2)*gw**2
                m_b = beta1*m_b + (1-beta1)*gb
                v_b = beta2*v_b + (1-beta2)*gb**2
                m_w_c = m_w / (1 - beta1**t_adam)
                v_w_c = v_w / (1 - beta2**t_adam)
                m_b_c = m_b / (1 - beta1**t_adam)
                v_b_c = v_b / (1 - beta2**t_adam)
                w -= lr * m_w_c / (np.sqrt(v_w_c) + eps_adam)
                b -= lr * m_b_c / (np.sqrt(v_b_c) + eps_adam)

        # Métricas al final de cada época
        p_tr = sigmoid(X_tr @ w + b)
        p_te = sigmoid(X_te @ w + b)
        hist['perdida_tr'].append(entropia_cruzada(y_tr, p_tr))
        hist['perdida_te'].append(entropia_cruzada(y_te, p_te))
        hist['auroc_te'].append(auroc_rapido(y_te, p_te))

    return w, b, hist


# Entrenar los tres optimizadores
configs = [
    ('GD (lotes completos)', 'gd',  0.5,  'steelblue'),
    ('SGD (mini-lotes=64)', 'sgd',  0.05, 'tomato'),
    ('Adam',                'adam', 0.01, 'seagreen'),
]

resultados = {}
for nombre, opt, lr, color in configs:
    w_f, b_f, hist = entrenar_logistico(
        X_tr, y_tr, X_te, y_te,
        optimizador=opt, lr=lr, n_epochs=150, rng_=rng
    )
    resultados[nombre] = (w_f, b_f, hist, color)
    print(f'{nombre:<25}  pérdida_te={hist["perdida_te"][-1]:.4f}  '
          f'AUROC={hist["auroc_te"][-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epocas = range(1, 151)

for nombre, (w_f, b_f, hist, color) in resultados.items():
    axes[0].plot(epocas, hist['perdida_tr'], lw=2, color=color,
                  label=nombre)
    axes[1].plot(epocas, hist['auroc_te'],   lw=2, color=color,
                  label=nombre)

axes[0].set(xlabel='Época', ylabel='Pérdida (entropía cruzada)',
            title='Convergencia del entrenamiento\nGD vs SGD vs Adam')
axes[0].legend(fontsize=9)
axes[1].set(xlabel='Época', ylabel='AUROC (test)',
            title='AUROC en test por época')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Parte 3 — Regularización L1 y L2: trayectorias y dispersión

La regularización añade un término de penalización a la pérdida:

$$\mathcal{L}_\text{reg}(\mathbf{w}) = \mathcal{L}(\mathbf{w}) + \lambda \Omega(\mathbf{w})$$

| Regularización | $\Omega(\mathbf{w})$ | Equivalencia bayesiana | Efecto |
|---|---|---|---|
| **L2 (ridge)** | $\|\mathbf{w}\|_2^2$ | Prior gaussiano | Shrinkage uniforme |
| **L1 (lasso)** | $\|\mathbf{w}\|_1$ | Prior de Laplace | Shrinkage + dispersión (ceros exactos) |

La diferencia geométrica es clave: la bola L2 es suave (los mínimos raramente tocan los ejes),
mientras que la bola L1 tiene esquinas en los ejes (los mínimos frecuentemente tocan los ejes → coeficientes exactamente cero).

In [ ]:
# ── Trayectorias de regularización L1 vs L2 ───────────────────────────────────
lambdas = np.logspace(-4, 1, 30)

coefs_l2 = []
coefs_l1 = []
aurocs_l2 = []
aurocs_l1 = []

for lam in lambdas:
    _, _, hist_l2 = entrenar_logistico(
        X_tr, y_tr, X_te, y_te,
        optimizador='adam', lr=0.01, n_epochs=200,
        lam=lam, penalizacion='l2', rng_=rng
    )
    w_l2, _, _ = entrenar_logistico(
        X_tr, y_tr, X_te, y_te,
        optimizador='adam', lr=0.01, n_epochs=200,
        lam=lam, penalizacion='l2', rng_=rng
    )
    _, _, hist_l1 = entrenar_logistico(
        X_tr, y_tr, X_te, y_te,
        optimizador='adam', lr=0.01, n_epochs=200,
        lam=lam, penalizacion='l1', rng_=rng
    )
    w_l1, _, _ = entrenar_logistico(
        X_tr, y_tr, X_te, y_te,
        optimizador='adam', lr=0.01, n_epochs=200,
        lam=lam, penalizacion='l1', rng_=rng
    )
    coefs_l2.append(w_l2)
    coefs_l1.append(w_l1)
    aurocs_l2.append(hist_l2['auroc_te'][-1])
    aurocs_l1.append(hist_l1['auroc_te'][-1])

# Simplificar: re-entrenar retornando los pesos correctamente
coefs_l2, coefs_l1, aurocs_l2, aurocs_l1 = [], [], [], []
for lam in lambdas:
    w2, b2, h2 = entrenar_logistico(X_tr, y_tr, X_te, y_te,
                                      optimizador='adam', lr=0.01,
                                      n_epochs=200, lam=lam,
                                      penalizacion='l2', rng_=rng)
    w1, b1, h1 = entrenar_logistico(X_tr, y_tr, X_te, y_te,
                                      optimizador='adam', lr=0.01,
                                      n_epochs=200, lam=lam,
                                      penalizacion='l1', rng_=rng)
    coefs_l2.append(w2)
    coefs_l1.append(w1)
    aurocs_l2.append(h2['auroc_te'][-1])
    aurocs_l1.append(h1['auroc_te'][-1])

coefs_l2 = np.array(coefs_l2)   # (n_lambdas, n_feats)
coefs_l1 = np.array(coefs_l1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colores_feats = plt.cm.tab10(np.linspace(0, 1, N_feats))

for j in range(N_feats):
    axes[0].plot(lambdas, coefs_l2[:, j], lw=2,
                  color=colores_feats[j], label=nombres_feats[j])
    axes[1].plot(lambdas, coefs_l1[:, j], lw=2,
                  color=colores_feats[j], label=nombres_feats[j])

axes[0].set(xscale='log', xlabel='λ', ylabel='Coeficiente',
            title='Trayectoria Ridge (L2)\nShrinkage uniforme — nunca exactamente cero')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].axhline(0, color='gray', lw=0.8)

axes[1].set(xscale='log', xlabel='λ', ylabel='Coeficiente',
            title='Trayectoria Lasso (L1)\nShrinkage + dispersión — coeficientes = 0 exacto')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].axhline(0, color='gray', lw=0.8)

axes[2].semilogx(lambdas, aurocs_l2, 'steelblue', lw=2, label='L2 (ridge)')
axes[2].semilogx(lambdas, aurocs_l1, 'tomato',    lw=2, label='L1 (lasso)')
axes[2].set(xlabel='λ', ylabel='AUROC (test)',
            title='AUROC vs fuerza de regularización')
axes[2].legend()

plt.suptitle('Regularización L1 vs L2 — Regresión logística en datos UCI',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ¿Cuántos coeficientes son exactamente cero con L1?
lam_idx = len(lambdas) // 2
ceros_l1 = np.sum(np.abs(coefs_l1[lam_idx]) < 0.01)
ceros_l2 = np.sum(np.abs(coefs_l2[lam_idx]) < 0.01)
print(f'Con λ={lambdas[lam_idx]:.4f}:')
print(f'  L1: {ceros_l1}/{N_feats} coeficientes ≈ 0  (dispersión)')
print(f'  L2: {ceros_l2}/{N_feats} coeficientes ≈ 0  (sin dispersión)')

## Parte 4 — Softmax multiclase: estados de gravedad en UCI

Para $K > 2$ clases, la regresión logística se extiende mediante la función **softmax**:

$$P(y=k \mid \mathbf{x}) = \frac{e^{\mathbf{w}_k^\top \mathbf{x} + b_k}}{\sum_{j=1}^{K} e^{\mathbf{w}_j^\top \mathbf{x} + b_j}}$$

La pérdida es la entropía cruzada categórica:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K} y_{ik}\log\hat{p}_{ik}$$

In [ ]:
# ── Softmax multiclase: 3 niveles de gravedad en UCI ─────────────────────────
# Clase 0: bajo riesgo (sobrevive sin complicaciones)
# Clase 1: riesgo intermedio (sobrevive con complicaciones)
# Clase 2: alto riesgo (fallece en UCI)

K = 3

def generar_uci_multiclase(N, rng_):
    X, _ = generar_uci(N, rng_)
    X_s  = (X - X.mean(0)) / (X.std(0) + 1e-8)
    # Apache II como proxy de gravedad
    apache_s = X_s[:, 1]
    logit_int = 0.8 * apache_s + rng_.normal(0, 0.5, N)
    logit_alt = 1.5 * apache_s + rng_.normal(0, 0.5, N)
    y_mc = np.where(logit_alt > 0.8, 2,
           np.where(logit_int > 0.0, 1, 0)).astype(int)
    return X_s.astype(np.float32), y_mc

X_mc, y_mc = generar_uci_multiclase(N, rng)
idx_tr_mc  = idx_tr
idx_te_mc  = idx_te
X_tr_mc, y_tr_mc = X_mc[idx_tr_mc], y_mc[idx_tr_mc]
X_te_mc, y_te_mc = X_mc[idx_te_mc], y_mc[idx_te_mc]

def softmax(Z):
    Z_s = Z - Z.max(axis=1, keepdims=True)   # estabilidad numérica
    E   = np.exp(Z_s)
    return E / E.sum(axis=1, keepdims=True)

def perdida_categorica(Y_oh, P):
    return -np.mean(np.sum(Y_oh * np.log(P + 1e-9), axis=1))

def exactitud(y_true, P):
    return (P.argmax(axis=1) == y_true).mean()

# One-hot encoding
def one_hot(y, K):
    Y = np.zeros((len(y), K))
    Y[np.arange(len(y)), y] = 1
    return Y

Y_tr_oh = one_hot(y_tr_mc, K)
Y_te_oh = one_hot(y_te_mc, K)

# Entrenamiento softmax con descenso de gradiente
d = X_tr_mc.shape[1]
W_soft = np.zeros((d, K))
b_soft = np.zeros(K)
lr_soft = 0.05

hist_soft = {'perdida': [], 'exactitud': []}

for epoch in range(300):
    P_tr = softmax(X_tr_mc @ W_soft + b_soft)
    # Gradientes
    delta    = (P_tr - Y_tr_oh) / len(y_tr_mc)
    grad_W   = X_tr_mc.T @ delta
    grad_b   = delta.sum(axis=0)
    W_soft  -= lr_soft * grad_W
    b_soft  -= lr_soft * grad_b

    if epoch % 10 == 0:
        P_te = softmax(X_te_mc @ W_soft + b_soft)
        hist_soft['perdida'].append(perdida_categorica(Y_te_oh, P_te))
        hist_soft['exactitud'].append(exactitud(y_te_mc, P_te))

print(f'Softmax multiclase (3 niveles de gravedad UCI):')
print(f'  Exactitud final en test: {hist_soft["exactitud"][-1]:.3f}')
print(f'  Pérdida final en test:   {hist_soft["perdida"][-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epocas_s = range(0, 300, 10)
axes[0].plot(epocas_s, hist_soft['perdida'],   'steelblue', lw=2)
axes[0].set(xlabel='Época', ylabel='Pérdida categórica', title='Convergencia softmax')
axes[1].plot(epocas_s, hist_soft['exactitud'], 'seagreen',  lw=2)
axes[1].axhline(1/K, color='gray', ls='--', lw=1, label=f'Azar ({1/K:.2f})')
axes[1].set(xlabel='Época', ylabel='Exactitud', title='Exactitud en test')
axes[1].legend()
plt.suptitle('Regresión logística multiclase (softmax) — 3 niveles de gravedad UCI',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Parte 5 — Curvas de aprendizaje: diagnóstico de sesgo/varianza

Las curvas de aprendizaje grafican el error de entrenamiento y validación
en función del **tamaño del conjunto de entrenamiento** (no de las épocas).
Son el primer diagnóstico a realizar antes de elegir un modelo más complejo.

In [ ]:
# ── Curvas de aprendizaje ─────────────────────────────────────────────────────
n_sizes  = [20, 40, 80, 150, 250, 400, len(X_tr)]
n_rep    = 10   # repeticiones por tamaño (para estimar varianza)

perdidas_tr_media = []
perdidas_te_media = []
aurocs_te_media   = []
perdidas_tr_std   = []
perdidas_te_std   = []

for n_s in n_sizes:
    p_tr_rep, p_te_rep, auc_rep = [], [], []
    for _ in range(n_rep):
        # Submuestrear manteniendo la proporción de clases
        n_p = max(5, round(n_s * y_tr.mean()))
        n_n = n_s - n_p
        idx_p_s = rng.choice(np.where(y_tr==1)[0], n_p, replace=False)
        idx_n_s = rng.choice(np.where(y_tr==0)[0], n_n, replace=False)
        idx_s   = np.concatenate([idx_p_s, idx_n_s])

        w_s, b_s, h_s = entrenar_logistico(
            X_tr[idx_s], y_tr[idx_s], X_te, y_te,
            optimizador='adam', lr=0.01, n_epochs=200, rng_=rng
        )
        p_tr_rep.append(h_s['perdida_tr'][-1])
        p_te_rep.append(h_s['perdida_te'][-1])
        auc_rep.append(h_s['auroc_te'][-1])

    perdidas_tr_media.append(np.mean(p_tr_rep))
    perdidas_te_media.append(np.mean(p_te_rep))
    aurocs_te_media.append(np.mean(auc_rep))
    perdidas_tr_std.append(np.std(p_tr_rep))
    perdidas_te_std.append(np.std(p_te_rep))

perdidas_tr_media = np.array(perdidas_tr_media)
perdidas_te_media = np.array(perdidas_te_media)
perdidas_tr_std   = np.array(perdidas_tr_std)
perdidas_te_std   = np.array(perdidas_te_std)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva de aprendizaje — pérdida
axes[0].plot(n_sizes, perdidas_tr_media, 'steelblue', lw=2.5, label='Entrenamiento')
axes[0].fill_between(n_sizes,
                      perdidas_tr_media - perdidas_tr_std,
                      perdidas_tr_media + perdidas_tr_std,
                      alpha=0.2, color='steelblue')
axes[0].plot(n_sizes, perdidas_te_media, 'tomato', lw=2.5, label='Validación (test)')
axes[0].fill_between(n_sizes,
                      perdidas_te_media - perdidas_te_std,
                      perdidas_te_media + perdidas_te_std,
                      alpha=0.2, color='tomato')
axes[0].set(xlabel='Tamaño del conjunto de entrenamiento',
            ylabel='Pérdida (entropía cruzada)',
            title='Curvas de aprendizaje — pérdida\n'
                  'Brecha grande → varianza alta | Ambas altas → sesgo alto')
axes[0].legend()

# AUROC vs N
axes[1].plot(n_sizes, aurocs_te_media, 'seagreen', lw=2.5, marker='o', ms=7)
axes[1].axhline(0.5, color='gray', ls='--', lw=1, label='Azar')
axes[1].set(xlabel='Tamaño del conjunto de entrenamiento',
            ylabel='AUROC (test)',
            title='AUROC vs tamaño muestral')
axes[1].legend()

plt.suptitle('Curvas de aprendizaje — Regresión logística UCI',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Diagnóstico
brecha_final = perdidas_te_media[-1] - perdidas_tr_media[-1]
print(f'Brecha final (test − train): {brecha_final:.4f}')
if brecha_final > 0.05:
    print('  → Varianza alta: el modelo se beneficiaría de más datos o regularización')
elif perdidas_te_media[-1] > 0.55:
    print('  → Sesgo alto: el modelo es demasiado simple para estos datos')
else:
    print('  → Modelo bien calibrado: sesgo y varianza equilibrados')

## ✏️ Ejercicios

Los ejercicios usan el dataset **WESAD** (Schmidt et al., 2018) — detección de estrés
fisiológico a partir de señales de pulsera (ECG, EDA, TEMP, ACC).

> **Fuente:** Schmidt, P. et al. (2018). Introducing WESAD, a multimodal dataset for
> wearable stress and affect detection. *ACM ICMI 2018*, 400–408.
> https://doi.org/10.1145/3242969.3242985

```python
# Dataset simulado con distribución WESAD (estrés vs línea base)
N_wesad = 500
y_wesad = rng.integers(0, 2, N_wesad)
X_wesad = np.column_stack([
    rng.normal(70 + 15*y_wesad, 12, N_wesad),   # FC (lpm)
    rng.normal(2  + 8 *y_wesad, 3,  N_wesad),   # EDA (μS)
    rng.normal(36 - 0.3*y_wesad, 0.4, N_wesad), # TEMP (°C)
    rng.normal(0.3 + 0.2*y_wesad, 0.1, N_wesad) # ACC (g)
]).astype(np.float32)
```

1. **Verificación del gradiente.** Implementa la verificación numérica del gradiente
   para la función `gradiente_logistico` usando la aproximación de diferencias finitas:
   $g_j \approx (\mathcal{L}(\mathbf{w}+\epsilon\mathbf{e}_j) - \mathcal{L}(\mathbf{w}-\epsilon\mathbf{e}_j))/(2\epsilon)$.
   La diferencia relativa debe ser $< 10^{-5}$ para $\epsilon=10^{-5}$.

2. **Comparación de optimizadores en WESAD.** Entrena la regresión logística en el dataset
   WESAD con GD, SGD y Adam. Para cada optimizador, grafica la pérdida y el AUROC por época.
   ¿Cuál converge más rápido? ¿Cuál obtiene mejor AUROC final?

3. **Selección de λ por validación cruzada.** Para el dataset WESAD, usa k-fold (k=5)
   para seleccionar el λ óptimo de regularización L2 en el rango $[10^{-4}, 10]$.
   Grafica el AUROC medio ± desviación estándar vs λ. ¿El λ óptimo es el mismo con L1?

4. **Regresión logística multinomial.** Extiende el clasificador de WESAD a 3 clases:
   línea base, estrés, y amusement. Añade una tercera clase al dataset simulado
   y entrena el modelo softmax. Reporta la matriz de confusión y el AUROC one-vs-rest
   para cada clase.

5. *(Desafío)* **Regresión logística con LOSO en WESAD real.** Descarga el dataset WESAD
   (https://uni-siegen.de/life/software/wesad.html). Implementa el pipeline completo
   con LOSO (15 sujetos): extracción de características HRV → normalización dentro del
   fold → selección de λ en el fold interno → evaluación en test. Reporta AUROC ± std
   y compara con la línea base de permutaciones.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| PhysioNet CinC 2012 (UCI) | Silva, I. et al. (2012). *Computing in Cardiology*, 39, 245–248. https://physionet.org/content/challenge-2012/ | Dataset principal del Módulo II |
| WESAD | Schmidt, P. et al. (2018). *ACM ICMI 2018*. https://doi.org/10.1145/3242969.3242985 | Ejercicios de la Sesión 06 |